In [22]:
import pandas as pd
import numpy as np
import jdatetime as jdt
import jalali_pandas

In [23]:
mydf = pd.read_excel('WoodInc-sale.xlsx')
mydf2 = pd.read_excel('WoodInc-purchase.xlsx')


In [24]:
mydf = pd.read_excel('WoodInc-sale.xlsx')
mydf.rename(columns={'تاريخ':'date','نام کالا':'productName','کد کالا':'productID','نام مشتري':'customerName','کد مشتري':'customerID','شماره':'docNumber','مبلغ خالص':'netAmount','ماليات':'tax','مبلغ':'amount','في':'unitPrice','مقدار':'quantity','واحد':'unit'}, inplace=True)
mydf.head(1)

,date,docNumber,customerID,customerName,productID,productName,unit,quantity,unitPrice,amount,tax,netAmount
0,13990105,82516,1101328,ابوالفضل تام ...,1-1-04-706,ملامينه 210*280 كد 361 سفيد صابوني كيميا ...,ورق ...,5.0,2339450,11697250,1052753,12750003


In [25]:
mydf2.columns

Index(['تاريخ', 'شماره', 'کد تأمین کننده', 'نام تأمین کننده', 'کد کالا',
       'نام کالا', 'واحد', 'مقدار', 'في', 'مبلغ', 'عوارض', 'مبلغ خالص'],
      dtype='str')

In [ ]:
mydf2 = pd.read_excel('WoodInc-purchase.xlsx')
mydf2.rename(columns={
    'تاريخ':'date' ,
    'شماره' :'docNumber',
    'کد تأمین کننده':'supplierID',
    'نام تأمین کننده' :'supplierName',
    'کد کالا'  : 'productID',
    'نام کالا' :'productName',
    'واحد'    :'unit',
    'مقدار'   : 'quantity',
    'في'    :   'unitPrice',
    'مبلغ'  :'amount',
    'عوارض' :'tax',
    'مبلغ خالص' :'netAmount'},inplace=True)
    
mydf2.head(2)

,date,docNumber,supplierID,supplierName,productID,productName,unite,quantity,unitePrice,amount,tax,netAmount
0,13990117,1,1100251,شركت تخته فشرده شمال ...,1-1-04-015,ملامينه 204*260 كد 1 سفيد درجه A ...,ورق ...,315.0,1879000,-591885000,0,-645154650
1,13990117,1,1100251,شركت تخته فشرده شمال ...,1-1-04-022,ملامينه 204*260 كد 78 آنتيك لايت درجه A ...,ورق ...,45.0,2087000,-93915000,0,-102367350


In [27]:
sale_gp = mydf.groupby(['productID','productName']).agg(total_sale_quantity=('quantity','sum')).reset_index()
sale_gp.head(2)

,productID,productName,total_sale_quantity
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,66.0
1,1-1-01-029,ام دي اف 210*366 كد 60 ونگه درجه A ...,125.0


In [28]:
purchase_gp = mydf2.groupby(['productID','productName']).agg(total_purchase_quantity=('quantity','sum')).reset_index()
purchase_gp.head(6)

,productID,productName,total_purchase_quantity
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,66.0
1,1-1-01-029,ام دي اف 210*366 كد 60 ونگه درجه A ...,68.0
2,1-1-01-034,ام دي اف 210*366 كد 71 *AF47* درجه A ...,39.0
3,1-1-01-050,ام دي اف 210*366 كد 110 درجه A ...,31.0
4,1-1-01-055,ام دي اف 210*366 كد 41 بلوط برفي درجه A ...,30.0
5,1-1-01-061,ام دي اف 210*366 كد 97 درجه A ...,46.0


In [29]:
inventory_year = pd.merge( left=purchase_gp ,right=sale_gp ,on=['productID','productName'],how='outer')
inventory_year.head()

,productID,productName,total_purchase_quantity,total_sale_quantity
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,66.0,66.0
1,1-1-01-029,ام دي اف 210*366 كد 60 ونگه درجه A ...,68.0,125.0
2,1-1-01-034,ام دي اف 210*366 كد 71 *AF47* درجه A ...,39.0,86.0
3,1-1-01-050,ام دي اف 210*366 كد 110 درجه A ...,31.0,35.0
4,1-1-01-053,ام دي اف 210*366 كد 13 نقره اي درجه A ...,NaN,8.0


In [30]:
inventory_year[['total_purchase_quantity','total_sale_quantity',]]=inventory_year[['total_purchase_quantity','total_sale_quantity']].fillna(0)
inventory_year.head()

,productID,productName,total_purchase_quantity,total_sale_quantity
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,66.0,66.0
1,1-1-01-029,ام دي اف 210*366 كد 60 ونگه درجه A ...,68.0,125.0
2,1-1-01-034,ام دي اف 210*366 كد 71 *AF47* درجه A ...,39.0,86.0
3,1-1-01-050,ام دي اف 210*366 كد 110 درجه A ...,31.0,35.0
4,1-1-01-053,ام دي اف 210*366 كد 13 نقره اي درجه A ...,0.0,8.0


موجودی پایان سال میشه تفاوت مقدار خرید و فروش 

In [31]:
inventory_year['end_year_inventory'] = (inventory_year['total_purchase_quantity']-inventory_year['total_sale_quantity'])
inventory_year.head(20)

,productID,productName,total_purchase_quantity,total_sale_quantity,end_year_inventory
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,66.0,66.0,0.0
1,1-1-01-029,ام دي اف 210*366 كد 60 ونگه درجه A ...,68.0,125.0,-57.0
2,1-1-01-034,ام دي اف 210*366 كد 71 *AF47* درجه A ...,39.0,86.0,-47.0
3,1-1-01-050,ام دي اف 210*366 كد 110 درجه A ...,31.0,35.0,-4.0
4,1-1-01-053,ام دي اف 210*366 كد 13 نقره اي درجه A ...,0.0,8.0,-8.0
5,1-1-01-055,ام دي اف 210*366 كد 41 بلوط برفي درجه A ...,30.0,81.0,-51.0
6,1-1-01-061,ام دي اف 210*366 كد 97 درجه A ...,46.0,9.0,37.0
7,1-1-01-064,ام دي اف 210*366 كد 85 لاركس درجه A ...,55.0,45.0,10.0
8,1-1-01-068,ام دي اف 210*366 كد 19 مشكي درجه A ...,53.0,53.0,0.0
9,1-1-01-099,ام دي اف 3 ميل 183*244 كد519برجسته آسپن ...,0.0,2.0,-2.0


سوال 4 ازم موجودی پایان هرماه را خواسته

In [32]:
def to_jalali_date(column, year_dir='L'):
    column = pd.Series(column)

    if column.dtype == 'object':
        sep = next((s for s in ['/', '-', '.'] if s in str(column.iloc[0])), None)
        mod_col = [x.split(sep) for x in column]
    
    elif pd.api.types.is_integer_dtype(column):
        str_col = column.astype(str)
        sample = str_col.iloc[0]
        if len(sample) == 8:
            if year_dir == 'L':
                mod_col = [[x[:4], x[4:6], x[6:]] for x in str_col]
            else:
                mod_col = [[x[:2], x[2:4], x[4:]] for x in str_col]
        elif len(sample) == 6:
            mod_col = [[x[:2], x[2:4], x[4:]] for x in str_col]
        else:
            raise ValueError("Unsupported integer date format")

    else:
        raise TypeError("Unsupported column dtype")

    # Extract year, month, day based on year direction
    idx = (0, 1, 2) if year_dir == 'L' else (2, 1, 0)
    year, month, day = zip(*[(i[idx[0]], i[idx[1]], i[idx[2]]) for i in mod_col])

    # Fix 2-digit years
    year = [
        f"14{y}" if len(y) == 2 and int(y) < 50 else
        f"13{y}" if len(y) == 2 and int(y) >= 50 else
        y
        for y in year
    ]

    return [jdt.date(int(y), int(m), int(d)) for y, m, d in zip(year, month, day)]

In [33]:
mydf ['jalaliDate'] = to_jalali_date(mydf['date'])

In [34]:
mydf2['jalaliDate'] = to_jalali_date(mydf2['date'])

In [35]:
mydf['month'] = mydf['jalaliDate'].jalali.month
mydf2['month'] = mydf2['jalaliDate'].jalali.month

In [36]:
sale_month_gp = mydf.groupby(
    ['month','productID','productName']
).agg(
    month_sale_quantity=('quantity','sum')
).reset_index()

In [37]:
purchase_month_gp = mydf2.groupby(
    ['month','productID','productName']
).agg(
    month_purchase_quantity=('quantity','sum')
).reset_index()

In [44]:
 month_inventory = pd.merge( left= purchase_month_gp,right=sale_month_gp,on=['month','productID','productName'], how='outer')
 month_inventory.head(4)

,month,productID,productName,month_purchase_quantity,month_sale_quantity
0,1,1-1-01-029,ام دي اف 210*366 كد 60 ونگه درجه A ...,NaN,6.0
1,1,1-1-01-034,ام دي اف 210*366 كد 71 *AF47* درجه A ...,NaN,9.0
2,1,1-1-01-050,ام دي اف 210*366 كد 110 درجه A ...,NaN,1.0
3,1,1-1-01-055,ام دي اف 210*366 كد 41 بلوط برفي درجه A ...,NaN,15.0


In [43]:
product = inventory_year [['productID','productName']] 
months= pd.DataFrame({'month': [1,2,3,4,5,6,7,8,9,10,11,12]})
product_months = pd.merge(left= product,right=months,how='cross')
product_months.head(15)

,productID,productName,month
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,1
1,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,2
2,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,3
3,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,4
4,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,5
5,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,6
6,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,7
7,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,8
8,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,9
9,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,10


In [ ]:
month_inventory_complete = pd.merge (
    left= product_months,
    right= month_inventory,
    how='left',
    on=['month', 'productID', 'productName']
)
month_inventory_complete.head(10)

,productID,productName,month,month_purchase_quantity,month_sale_quantity
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,1,NaN,NaN
1,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,2,NaN,NaN
2,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,3,NaN,NaN
3,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,4,66.0,66.0
4,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,5,NaN,NaN
5,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,6,NaN,NaN
6,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,7,NaN,NaN
7,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,8,NaN,NaN
8,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,9,NaN,NaN
9,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,10,NaN,NaN


In [ ]:
month_inventory_complete[['month_purchase_quantity','month_sale_quantity']] = month_inventory_complete[['month_purchase_quantity','month_sale_quantity']].fillna(0)
month_inventory_complete.head(10)

,productID,productName,month,month_purchase_quantity,month_sale_quantity
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,1,0.0,0.0
1,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,2,0.0,0.0
2,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,3,0.0,0.0
3,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,4,66.0,66.0
4,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,5,0.0,0.0
5,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,6,0.0,0.0
6,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,7,0.0,0.0
7,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,8,0.0,0.0
8,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,9,0.0,0.0
9,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,10,0.0,0.0


In [ ]:
month_inventory_complete ['month_inventory_change'] = (month_inventory_complete['month_purchase_quantity']-month_inventory_complete['month_sale_quantity'])
month_inventory_complete.head(10)

,productID,productName,month,month_purchase_quantity,month_sale_quantity,month_inventory_change
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,1,0.0,0.0,0.0
1,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,2,0.0,0.0,0.0
2,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,3,0.0,0.0,0.0
3,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,4,66.0,66.0,0.0
4,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,5,0.0,0.0,0.0
5,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,6,0.0,0.0,0.0
6,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,7,0.0,0.0,0.0
7,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,8,0.0,0.0,0.0
8,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,9,0.0,0.0,0.0
9,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,10,0.0,0.0,0.0


In [ ]:
month_inventory_complete = month_inventory_complete.sort_values( by= ['productID','month']).reset_index(drop=True)
month_inventory_complete.head(10)

,productID,productName,month,month_purchase_quantity,month_sale_quantity,month_inventory_change
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,1,0.0,0.0,0.0
1,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,2,0.0,0.0,0.0
2,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,3,0.0,0.0,0.0
3,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,4,66.0,66.0,0.0
4,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,5,0.0,0.0,0.0
5,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,6,0.0,0.0,0.0
6,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,7,0.0,0.0,0.0
7,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,8,0.0,0.0,0.0
8,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,9,0.0,0.0,0.0
9,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,10,0.0,0.0,0.0


In [ ]:
month_inventory_complete ['end_month_inventory'] = (month_inventory_complete.groupby('productID')['month_inventory_change'].cumsum())
month_inventory_complete

,productID,productName,month,month_purchase_quantity,month_sale_quantity,month_inventory_change,end_month_inventory
0,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,1,0.0,0.0,0.0,0.0
1,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,2,0.0,0.0,0.0,0.0
2,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,3,0.0,0.0,0.0,0.0
3,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,4,66.0,66.0,0.0,0.0
4,1-1-01-004,ام دي اف 183*366 سفيد يكرو مغز تايلندي درجه 1 ...,5,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
9247,1-1-12-031,صفحه كابينت 4.8 سانتي 60 يك لب كد 116 سامان يو...,8,0.0,3.0,-3.0,18.0
9248,1-1-12-031,صفحه كابينت 4.8 سانتي 60 يك لب كد 116 سامان يو...,9,0.0,3.0,-3.0,15.0
9249,1-1-12-031,صفحه كابينت 4.8 سانتي 60 يك لب كد 116 سامان يو...,10,0.0,4.0,-4.0,11.0
9250,1-1-12-031,صفحه كابينت 4.8 سانتي 60 يك لب كد 116 سامان يو...,11,0.0,0.0,0.0,11.0
